#Project Setup

# AI Customer Support Agent

## Project Overview

This project implements an AI-powered Customer Support Agent using Retrieval-Augmented Generation (RAG).

The chatbot answers customer queries by retrieving information from a product knowledge base stored in Pinecone and generating responses using OpenAI models.

## Objectives

- Build an intelligent customer support chatbot
- Retrieve accurate product information using Pinecone
- Answer complex customer queries using RAG
- Recommend alternative products when requested items are unavailable
- Maintain conversation context
- Deploy the application using Streamlit

## Technology Stack

- Python
- Google Colab
- OpenAI
- LangChain
- LangGraph
- Pinecone
- Streamlit

## Project Workflow

1. Install required libraries
2. Load API keys
3. Connect to Pinecone
4. Load product knowledge base
5. Retrieve relevant documents
6. Generate AI responses
7. Launch the Streamlit chatbot

#Install Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install -q --upgrade openai streamlit langchain langchain-core langchain-community langchain-openai langchain-experimental langgraph pinecone crewai crewai-tools pymupdf pypdf tiktoken python-dotenv pandas numpy

### Note on Library Incompatibility:

If you encountered an `ImportError` related to `langchain_core` or `langchain_community` after running the `!pip install` command, it is likely due to a version mismatch that isn't fully resolved until the Python runtime is reset.

**To resolve this:**
1. Go to the Colab menu: **Runtime > Restart runtime...**
2. After the runtime restarts, **run all cells from the beginning** (Cell > Run all).

This will ensure that all the newly installed and upgraded libraries are correctly loaded into the environment.

#Import Libraries

In [ ]:
# ==============================================================================
# 2. Consolidated Library Imports
# ==============================================================================

# Standard Libraries
import os
import glob
import warnings

# Google Colab Environment Secrets
from google.colab import userdata, files

# Data & Array Processing
import pandas as pd
import numpy as np

# Core OpenAI client
from openai import OpenAI

# LangChain Ecosystem Components
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker

# LangGraph (State management)
from langgraph.graph import StateGraph, START, END

# Pinecone (Vector database client)
from pinecone import Pinecone, ServerlessSpec

# CrewAI (Multi-agent framework)
from crewai import Agent, Task, Crew, Process

# PDF Parsing Engine
import fitz  # PyMuPDF

# Suppress annoying deprecation/cleanliness warnings
warnings.filterwarnings("ignore")

print("✅ All core libraries and sub-modules imported successfully!")

#Load API Keys

In [ ]:
# ============================================
# Load API Keys
# ============================================

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

# Optional
try:
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except:
    pass

print("✅ API keys loaded successfully!")

#Chunking

In [ ]:
#!pip install -q -U langchain-experimental

In [ ]:
#pip install --upgrade langchain langchain-core langchain-community

In [ ]:
#pip install --upgrade langchain-core langchain-community

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import PyPDFLoader

import os
import glob

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Quick check to ensure it initializes without crashing
print("Import successful!")

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [ ]:
semantic_chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
)

In [ ]:
pdf_folder = "documents"

all_documents = []

for pdf in glob.glob(f"{pdf_folder}/*.pdf"):
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    all_documents.extend(docs)

print(f"Loaded {len(all_documents)} pages.")

In [ ]:
semantic_chunks = semantic_chunker.split_documents(all_documents)

print(f"Total Semantic Chunks: {len(semantic_chunks)}")

In [ ]:
for i, chunk in enumerate(semantic_chunks):
    chunk.metadata["chunk_id"] = i + 1
    chunk.metadata["document_type"] = (
        chunk.metadata["source"]
        .split("/")[-1]
        .replace(".pdf", "")
    )

In [ ]:
for chunk in semantic_chunks[:5]:
    print("=" * 80)
    print(f"Chunk ID      : {chunk.metadata['chunk_id']}")
    print(f"Source        : {chunk.metadata['source']}")
    print(f"Page          : {chunk.metadata['page']}")
    print(f"Document Type : {chunk.metadata['document_type']}")
    print("-" * 80)
    print(chunk.page_content[:500])

In [ ]:
# ============================================
# Enhance Chunk Metadata
# ============================================

import os

# -----------------------------------------
# Product List
# -----------------------------------------
products = [
    "Ruby Solitaire Ring",
    "Emerald Halo Ring",
    "Diamond Eternity Ring",
    "Pearl Pendant Necklace",
    "Sapphire Tennis Bracelet",
    "Rose Gold Heart Pendant",
    "Diamond Stud Earrings",
    "Emerald Drop Earrings"
]

# -----------------------------------------
# Document IDs
# -----------------------------------------
DOC_IDS = {
    "Product_Catalog": "DOC001",
    "FAQ": "DOC002",
    "Return_Policy": "DOC003",
    "Shipping_Policy": "DOC004",
    "Warranty_Policy": "DOC005",
    "Jewellery_Care_Guide": "DOC006"
}

# -----------------------------------------
# Importance
# -----------------------------------------
IMPORTANCE = {
    "Product": "High",
    "Policy": "High",
    "FAQ": "Medium",
    "Care Guide": "Low",
    "General": "Low"
}

# -----------------------------------------
# Keywords
# -----------------------------------------
KEYWORDS = {

    "Product_Catalog": [
        "ruby","emerald","diamond","pearl","sapphire",
        "gold","white gold","yellow gold","rose gold",
        "silver","platinum","ring","bracelet",
        "necklace","earrings","solitaire","halo",
        "tennis bracelet","stud earrings","pendant"
    ],

    "Warranty_Policy": [
        "warranty","lifetime warranty","manufacturing defect",
        "repair","inspection","cleaning","loose gemstone",
        "clasp","scratches","chemical damage"
    ],

    "Return_Policy": [
        "return","exchange","refund","invoice",
        "original packaging","customized jewellery",
        "damaged product","business days"
    ],

    "Shipping_Policy": [
        "shipping","delivery","tracking",
        "metro cities","remote areas",
        "courier","free shipping",
        "shipping charges","dispatch"
    ],

    "FAQ": [
        "payment","credit card","debit card",
        "upi","emi","cash on delivery",
        "international shipping",
        "gift wrapping",
        "customer support"
    ],

    "Jewellery_Care_Guide": [
        "cleaning","storage","microfiber cloth",
        "soap","perfume","chlorine",
        "bleach","ultrasonic cleaner",
        "professional cleaning",
        "ring resizing"
    ]
}

# ============================================
# Process Each Chunk
# ============================================

for i, chunk in enumerate(semantic_chunks):

    # -----------------------------------------
    # Basic Information
    # -----------------------------------------
    source_file = os.path.basename(chunk.metadata.get("source", ""))
    document_name = source_file.replace(".pdf", "")
    document_id = DOC_IDS.get(document_name)

    # -----------------------------------------
    # Category
    # -----------------------------------------
    if "Product" in document_name:
        category = "Product"

    elif "FAQ" in document_name:
        category = "FAQ"

    elif "Return" in document_name:
        category = "Policy"

    elif "Shipping" in document_name:
        category = "Policy"

    elif "Warranty" in document_name:
        category = "Policy"

    elif "Care" in document_name:
        category = "Care Guide"

    else:
        category = "General"

    importance = IMPORTANCE.get(category)

    # -----------------------------------------
    # Product Name
    # -----------------------------------------
    product_name = None

    if category == "Product":
        for product in products:
            if product.lower() in chunk.page_content.lower():
                product_name = product
                break

    # -----------------------------------------
    # Section Detection
    # -----------------------------------------
    text = chunk.page_content.lower()

    if category == "Product":

        if "ruby solitaire ring" in text:
            section = "Ruby Solitaire Ring"

        elif "emerald halo ring" in text:
            section = "Emerald Halo Ring"

        elif "diamond eternity ring" in text:
            section = "Diamond Eternity Ring"

        elif "pearl pendant necklace" in text:
            section = "Pearl Pendant Necklace"

        elif "sapphire tennis bracelet" in text:
            section = "Sapphire Tennis Bracelet"

        elif "rose gold heart pendant" in text:
            section = "Rose Gold Heart Pendant"

        elif "diamond stud earrings" in text:
            section = "Diamond Stud Earrings"

        elif "emerald drop earrings" in text:
            section = "Emerald Drop Earrings"

        else:
            section = "Product Overview"

    elif document_name == "Warranty_Policy":

        if "lifetime warranty" in text:
            section = "Lifetime Warranty"

        elif "standard warranty" in text:
            section = "Standard Warranty"

        elif "warranty covers" in text:
            section = "Warranty Coverage"

        elif "does not cover" in text:
            section = "Warranty Exclusions"

        elif "cleaning" in text:
            section = "Cleaning Service"

        else:
            section = "Warranty"

    elif document_name == "Return_Policy":

        if "refund" in text:
            section = "Refund Policy"

        elif "exchange" in text:
            section = "Exchange Policy"

        else:
            section = "Return Policy"

    elif document_name == "Shipping_Policy":

        if "shipping charges" in text:
            section = "Shipping Charges"

        elif "delivery time" in text:
            section = "Delivery Time"

        elif "tracking" in text:
            section = "Order Tracking"

        else:
            section = "Shipping"

    elif document_name == "Jewellery_Care_Guide":

        if "professional cleaning" in text:
            section = "Professional Cleaning"

        elif "storage" in text:
            section = "Storage"

        elif "cleaning" in text:
            section = "Cleaning"

        elif "avoid" in text:
            section = "Precautions"

        else:
            section = "Jewellery Care"

    elif category == "FAQ":
        section = "Frequently Asked Questions"

    else:
        section = "General"

    # -----------------------------------------
    # Keywords
    # -----------------------------------------
    keywords = []

    for keyword in KEYWORDS.get(document_name, []):
        if keyword.lower() in text:
            keywords.append(keyword)

    if product_name:
        keywords.append(product_name)

    keywords = list(set(keywords))

    # -----------------------------------------
    # Remove Unnecessary Metadata
    # -----------------------------------------
    for field in [
        "producer",
        "creator",
        "creationdate",
        "author",
        "moddate"
    ]:
        chunk.metadata.pop(field, None)

    # -----------------------------------------
    # Update Metadata
    # -----------------------------------------
    chunk.metadata.update({

        "chunk_id": i + 1,
        "document_id": document_id,
        "document_type": document_name,
        "document_name": document_name,
        "source_file": source_file,
        "category": category,
        "importance": importance,
        "section": section,
        "product_name": product_name,
        "keywords": keywords,
        "page_number": chunk.metadata.get("page", 0)

    })

print(f"✅ Metadata enhanced for {len(semantic_chunks)} chunks.")

In [ ]:
semantic_chunks[2].metadata

In [ ]:
for chunk in semantic_chunks[:25]:
    print("=" * 80)
    print(chunk.metadata)

In [ ]:
print(f"Total Chunks: {len(semantic_chunks)}")

In [ ]:
import pandas as pd

metadata_list = []

for chunk in semantic_chunks:
    metadata_list.append(chunk.metadata)

metadata_df = pd.DataFrame(metadata_list)

metadata_df

In [ ]:
with open("metadata_report.txt", "w", encoding="utf-8") as f:
    f.write(metadata_df.to_string(index=False))

print("✅ metadata_report.txt created successfully.")

In [ ]:
from google.colab import files

files.download("metadata_report.txt")

In [ ]:
sample_text = "Returns are accepted within 30 days."

embedding = embeddings.embed_query(sample_text)

print(f"Embedding Dimension: {len(embedding)}")
print(embedding[:10])   # Print first 10 values

In [ ]:
print(semantic_chunks[0].page_content)

#Embedding

In [ ]:
print(f"Total chunks: {len(semantic_chunks)}")

# Display the first chunk
print("\nFirst Chunk:")
print(semantic_chunks[0].page_content[:300])

print("\nMetadata:")
print(semantic_chunks[0].metadata)

In [ ]:
texts = [doc.page_content for doc in semantic_chunks]

In [ ]:
filtered_chunks = [
    doc for doc in semantic_chunks
    if doc.page_content.strip()
]

print(f"Original chunks : {len(semantic_chunks)}")
print(f"Filtered chunks : {len(filtered_chunks)}")

In [ ]:
texts = [doc.page_content for doc in filtered_chunks]

In [ ]:
vectors = embeddings.embed_documents(texts)

In [ ]:
print(f"Number of chunks     : {len(texts)}")
print(f"Number of embeddings : {len(vectors)}")

In [ ]:
print(f"Embedding dimension: {len(vectors[0])}")

In [ ]:
print(vectors[0][:10])

In [ ]:
for i in range(3):
    print("=" * 60)
    print(f"Chunk {i}")
    print(filtered_chunks[i].metadata)
    print(filtered_chunks[i].page_content[:150])

In [ ]:
assert len(filtered_chunks) == len(vectors), \
    "Mismatch between chunks and embeddings."

assert len(vectors[0]) == 1536, \
    "Unexpected embedding dimension."

print("Embedding generation completed successfully.")

In [ ]:
print(type(texts))
print(type(vectors))
print(type(filtered_chunks))

#Connect to Pinecone

In [ ]:
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

In [ ]:
pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

In [ ]:
INDEX_NAME = "customer-support-agent"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 1536
METRIC = "cosine"
CLOUD_PROVIDER = "aws"
REGION = "us-east-1"

In [ ]:
existing_indexes = pc.list_indexes().names()
print(existing_indexes)

In [ ]:
if INDEX_NAME not in existing_indexes:

    print(f"Creating index: {INDEX_NAME}")

    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(
            cloud=CLOUD_PROVIDER,
            region=REGION
        )
    )

else:

    print(f"Index '{INDEX_NAME}' already exists.")

In [ ]:
while not pc.describe_index(INDEX_NAME).status["ready"]:
    print("Waiting for index to become ready...")
    time.sleep(5)

print("Index is ready.")

In [ ]:
index = pc.Index(INDEX_NAME)
print("Connected successfully.")

In [ ]:
index_info = pc.describe_index(INDEX_NAME)
print(index_info)

In [ ]:
stats = index.describe_index_stats()
print(stats)

In [ ]:
PINECONE_CONFIG = {
    "index_name": INDEX_NAME,
    "embedding_model": EMBEDDING_MODEL,
    "dimension": EMBEDDING_DIMENSION,
    "metric": METRIC,
    "cloud": CLOUD_PROVIDER,
    "region": REGION
}
print(PINECONE_CONFIG)

In [ ]:
print(f"Texts      : {len(texts)}")
print(f"Vectors    : {len(vectors)}")
print(f"Documents  : {len(filtered_chunks)}")

assert len(texts) == len(vectors) == len(filtered_chunks)

In [ ]:
vector_ids = [
    f"chunk_{i:05d}"
    for i in range(len(filtered_chunks))
]

print(vector_ids[:5])

In [ ]:
clean_metadata = []
for doc in filtered_chunks:
    metadata = {}
    for key, value in doc.metadata.items():
        if value is None:
            continue
        metadata[key] = str(value)
    clean_metadata.append(metadata)

In [ ]:
print(clean_metadata[0])

In [ ]:
for metadata, text in zip(clean_metadata, texts):

    metadata["text"] = text

In [ ]:
print(clean_metadata[0].keys())

In [ ]:
pinecone_records = []

for vector_id, embedding, metadata in zip(
    vector_ids,
    vectors,
    clean_metadata
):

    pinecone_records.append(
        (
            vector_id,
            embedding,
            metadata
        )
    )

In [ ]:
print(pinecone_records[0][0])
print(len(pinecone_records[0][1]))
print(pinecone_records[0][2])

In [ ]:
BATCH_SIZE = 100

In [ ]:
from math import ceil

total_batches = ceil(len(pinecone_records) / BATCH_SIZE)

for batch_number in range(total_batches):

    start = batch_number * BATCH_SIZE
    end = start + BATCH_SIZE

    batch = pinecone_records[start:end]

    index.upsert(vectors=batch)

    print(
        f"Uploaded batch "
        f"{batch_number + 1}/{total_batches}"
    )

In [ ]:
stats = index.describe_index_stats()

print(stats)

In [ ]:
uploaded_vectors = stats["total_vector_count"]

expected_vectors = len(filtered_chunks)

print(f"Uploaded : {uploaded_vectors}")
print(f"Expected : {expected_vectors}")

assert uploaded_vectors == expected_vectors

In [ ]:
query_vector = vectors[0]

results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

print(results)

In [ ]:
RAG_CONFIG = {

    "index_name": INDEX_NAME,

    "embedding_model": EMBEDDING_MODEL,

    "dimension": EMBEDDING_DIMENSION,

    "metric": METRIC,

    "batch_size": BATCH_SIZE
}

print(RAG_CONFIG)

#Build RAG Pipeline

#Customer Support Agent

#Streamlit Interface

#Testing